In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable
from pyspark.sql.window import Window
from datetime import datetime

In [0]:
%run ./UDF/udf_silver_incremental_ingest

In [0]:


src_bronze_path = "/Volumes/data_governance/bronze_access_management/audit_logs"
tgt_silver_table = "data_governance.silver_access_management.audit_logs"



In [0]:
df=silver_incremental_ingest(src_bronze_path,tgt_silver_table)

In [0]:
if df.count()==0:
    dbutils.notebook.exit("No new records to load")
else:
    pass

In [0]:
df.limit(20).display()

In [0]:
df = df.withColumn(
    "event_timestamp",
    to_timestamp("event_time")
)

df = df.withColumn("event_year", year("event_timestamp"))
df = df.withColumn("event_month", month("event_timestamp"))
df = df.withColumn("event_day", dayofmonth("event_timestamp"))
df = df.withColumn("event_hour", hour("event_timestamp"))

df = df.withColumn("service_name", upper(trim(col("service_name"))))
df = df.withColumn("action_name", upper(trim(col("action_name"))))
df = df.withColumn("audit_level", upper(trim(col("audit_level"))))




In [0]:
df = df.withColumn("run_by", col("identity_metadata.run_by")) \
       .withColumn("run_as", col("identity_metadata.run_as")) \
       .withColumn("acting_resource", col("identity_metadata.acting_resource")) \
       .withColumn("run_by_display_name", col("identity_metadata.run_by_display_name")) \
       .withColumn("run_as_display_name", col("identity_metadata.run_as_display_name"))


df = df.withColumn("user_email", col("user_identity.email")) \
       .withColumn("user_subject_name", col("user_identity.subject_name"))


df = df.withColumn("catalog_name", col("request_params.catalog_name")) \
       .withColumn("schema_name", col("request_params.schema_name")) \
       .withColumn("max_results", col("request_params.max_results")) \
       .withColumn("workspace_id_req", col("request_params.workspace_id")) \
       .withColumn("include_browse", col("request_params.include_browse")) \
       .withColumn("metastore_id", col("request_params.metastore_id"))


df = df.withColumn("status_code", col("response.status_code")) \
       .withColumn("error_message", col("response.error_message")) \
       .withColumn("response_result", col("response.result"))


df = df.drop(
    "request_params",
    "response",
    "identity_metadata",
    "user_identity"
)
df = df.withColumn("max_results", col("max_results").cast("int"))
df = df.withColumn("include_browse", col("include_browse").cast("boolean"))
df = df.withColumn("status_code", col("status_code").cast("int"))

In [0]:
df = df.withColumn(
    "is_success",
    col("status_code") == 200
)
df = df.withColumn(
    "is_system_user",
    col("user_email") == "System-User"
)

df = df.withColumn(
    "client_type",
    when(col("user_agent").contains("Mozilla"), "WEB_BROWSER")
    .when(col("user_agent").contains("Databricks-Service"), "DATABRICKS_INTERNAL")
    .when(col("user_agent").contains("Delta-Sharing"), "DELTA_SHARING_CLIENT")
    .otherwise("OTHER")
)

df = df.withColumn(
    "is_internal_ip",
    col("source_ip_address").startswith("172.")
)

df = df.withColumn(
    "source_ip_address",
    when(col("source_ip_address") == "", None).otherwise(col("source_ip_address"))
)

In [0]:
df.display()

In [0]:


df.write\
 .format("delta")\
 .mode("append") \
 .partitionBy("event_year", "event_month", "event_day") \
 .saveAsTable(tgt_silver_table)

In [0]:
%sql
select * from data_governance.silver_access_management.audit_logs;